[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_SpellCheck.ipynb)

# Benchmark: Spell Checking

Scores a pretrained spell checker's correction accuracy against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="spellcheck")`.

**Dataset**: [Peter Norvig's spelling test set](https://norvig.com/spell-testset1.txt) --
`correct: misspelling1 misspelling2 ...` pairs, the exact test data Norvig used to evaluate the
classic statistical spell-correction approach that `spellcheck_norvig` itself implements.

**Model**: `NorvigSweetingModel.pretrained("spellcheck_norvig")`.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, NorvigSweetingModel
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import urllib.request

url = "https://norvig.com/spell-testset1.txt"
raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

rows = []
for line in raw.splitlines():
    if not line.strip() or ":" not in line:
        continue
    correct, misspellings = line.split(":", 1)
    correct = correct.strip()
    for misspelled in misspellings.split():
        rows.append((misspelled, correct))

print(len(rows), "(misspelled, correct) pairs")
print(rows[:5])

270 (misspelled, correct) pairs
[('contenpted', 'contented'), ('contende', 'contented'), ('contended', 'contented'), ('contentid', 'contented'), ('begining', 'beginning')]

In [9]:
gold_data = spark.createDataFrame(rows, ["text", "label"])

## 2. Build the pipeline

In [11]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")
spell_checker = NorvigSweetingModel.pretrained("spellcheck_norvig") \
    .setInputCols(["token"]).setOutputCol("corrected")

pipeline = Pipeline(stages=[document_assembler, tokenizer, spell_checker])
pipeline_model = pipeline.fit(gold_data)

spellcheck_norvig download started this may take some time.
Approximate size to download 4.2 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[OK!]

> **Note: `predicted_col` disambiguates same-typed columns.** A spell checker's corrected
> output is still annotator type `token` -- the same type a plain `Tokenizer` produces -- so
> when a pipeline has both (as here, since we need a `Tokenizer` stage before the spell checker
> can run), `Benchmark.evaluate` can't tell them apart automatically. We pass `predicted_col`
> explicitly to say which one to score.

## 3. Run the benchmark

In [14]:
# Tokenizer and NorvigSweetingModel both emit annotatorType "token", so the default resolution
# would already pick "corrected" (the last one). Naming it explicitly is the escape hatch for
# when a pipeline's stage order doesn't put the column you want last.
report = Benchmark.evaluate(
    pipeline_model, gold_data, task="spellcheck", label_col="label", predicted_col="corrected")
print(report)

spellcheck accuracy (n=270): accuracy=0.5333, weightedF1=0.6011, weightedPrecision=0.7426, weightedRecall=0.5333
  access: f1=0.0000, precision=0.0000, recall=0.0000
  accessing: f1=0.0000, precision=0.0000, recall=0.0000
  accommodation: f1=0.8000, precision=1.0000, recall=0.6667
  account: f1=0.0000, precision=0.0000, recall=0.0000
  accusing: f1=0.0000, precision=0.0000, recall=0.0000
  aces: f1=0.0000, precision=0.0000, recall=0.0000
  acomodation: f1=0.0000, precision=0.0000, recall=0.0000
  acres: f1=0.0000, precision=0.0000, recall=0.0000
  address: f1=0.6667, precision=1.0000, recall=0.5000
  addressable: f1=1.0000, precision=1.0000, recall=1.0000
  amount: f1=0.0000, precision=0.0000, recall=0.0000
  ant: f1=0.0000, precision=0.0000, recall=0.0000
  arraigned: f1=0.0000, precision=0.0000, recall=0.0000
  arranged: f1=0.6667, precision=1.0000, recall=0.5000
  arrangeing: f1=0.0000, precision=0.0000, recall=0.0000
  arrangement: f1=0.0000, precision=0.0000, recall=0.0000
  arran